# Day 10: Capstone Project - Complete Data Analysis Pipeline

Today we'll build a complete end-to-end data analysis project using all the pandas skills learned over the past 9 days.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Capstone Project: E-commerce Sales Analysis")
print("=" * 50)

## Project Overview

**Business Problem**: An e-commerce company wants to understand their sales performance, customer behavior, and product trends to make data-driven decisions.

**Objectives**:
1. Analyze sales trends and seasonality
2. Identify top-performing products and categories
3. Segment customers based on purchasing behavior
4. Detect anomalies and outliers
5. Create actionable business insights
6. Build automated reporting system

## 1. Data Generation and Setup

In [ ]:
# Generate realistic e-commerce dataset
np.random.seed(42)

# Date range: 2 years of data
start_date = datetime(2022, 1, 1)
end_date = datetime(2023, 12, 31)
date_range = pd.date_range(start_date, end_date, freq='H')

# Product catalog
products = {
    'Electronics': ['Laptop', 'Smartphone', 'Tablet', 'Headphones', 'Camera'],
    'Clothing': ['T-Shirt', 'Jeans', 'Dress', 'Jacket', 'Shoes'],
    'Home': ['Sofa', 'Table', 'Lamp', 'Rug', 'Mirror'],
    'Books': ['Fiction', 'Non-Fiction', 'Textbook', 'Comic', 'Magazine'],
    'Sports': ['Basketball', 'Tennis Racket', 'Yoga Mat', 'Dumbbells', 'Bicycle']
}

# Generate transactions
n_transactions = 50000
n_customers = 5000

# Create base data
data = []
for i in range(n_transactions):
    # Select random category and product
    category = np.random.choice(list(products.keys()))
    product = np.random.choice(products[category])
    
    # Generate transaction details
    transaction = {
        'transaction_id': f'TXN_{i+1:06d}',
        'customer_id': f'CUST_{np.random.randint(1, n_customers+1):05d}',
        'product_name': product,
        'category': category,
        'quantity': np.random.poisson(2) + 1,
        'unit_price': np.random.lognormal(np.log(50), 0.8),
        'discount_percent': np.random.beta(2, 8) * 0.5,
        'timestamp': np.random.choice(date_range),
        'customer_age': np.random.normal(35, 12),
        'customer_gender': np.random.choice(['M', 'F'], p=[0.48, 0.52]),
        'payment_method': np.random.choice(['Credit Card', 'Debit Card', 'PayPal', 'Cash'], 
                                         p=[0.4, 0.3, 0.25, 0.05]),
        'shipping_cost': np.random.exponential(5),
        'customer_rating': np.random.choice([1, 2, 3, 4, 5], p=[0.02, 0.03, 0.15, 0.35, 0.45])
    }
    data.append(transaction)

# Create DataFrame
df = pd.DataFrame(data)

# Data preprocessing
df['customer_age'] = df['customer_age'].clip(18, 80).round().astype(int)
df['unit_price'] = df['unit_price'].round(2)
df['shipping_cost'] = df['shipping_cost'].round(2)
df['discount_amount'] = (df['unit_price'] * df['quantity'] * df['discount_percent']).round(2)
df['total_amount'] = (df['unit_price'] * df['quantity'] - df['discount_amount'] + df['shipping_cost']).round(2)

# Add seasonal effects
df['month'] = df['timestamp'].dt.month
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['hour'] = df['timestamp'].dt.hour
df['is_weekend'] = df['day_of_week'].isin([5, 6])

# Add some realistic business logic
# Higher prices for electronics
electronics_mask = df['category'] == 'Electronics'
df.loc[electronics_mask, 'unit_price'] *= 3
df.loc[electronics_mask, 'total_amount'] = (df.loc[electronics_mask, 'unit_price'] * 
                                           df.loc[electronics_mask, 'quantity'] - 
                                           df.loc[electronics_mask, 'discount_amount'] + 
                                           df.loc[electronics_mask, 'shipping_cost']).round(2)

print(f"Generated dataset with {len(df):,} transactions")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Number of unique customers: {df['customer_id'].nunique():,}")
print(f"Number of categories: {df['category'].nunique()}")
print(f"\nDataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 2. Data Quality Assessment

In [ ]:
# Comprehensive data quality check
def comprehensive_data_quality_check(df):
    print("DATA QUALITY ASSESSMENT")
    print("=" * 40)
    
    # Basic info
    print(f"Dataset shape: {df.shape}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print()
    
    # Missing values
    missing_values = df.isnull().sum()
    if missing_values.sum() > 0:
        print("Missing values:")
        print(missing_values[missing_values > 0])
    else:
        print("✓ No missing values found")
    print()
    
    # Duplicates
    duplicates = df.duplicated().sum()
    print(f"Duplicate rows: {duplicates} ({duplicates/len(df)*100:.2f}%)")
    
    # Data types
    print("\nData types:")
    print(df.dtypes)
    
    # Outliers in numerical columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    print(f"\nOutlier analysis for numerical columns:")
    
    for col in numerical_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        if outliers > 0:
            print(f"{col}: {outliers} outliers ({outliers/len(df)*100:.2f}%)")
    
    # Business logic validation
    print("\nBusiness logic validation:")
    
    # Check for negative values where they shouldn't exist
    negative_checks = {
        'quantity': (df['quantity'] <= 0).sum(),
        'unit_price': (df['unit_price'] <= 0).sum(),
        'total_amount': (df['total_amount'] <= 0).sum()
    }
    
    for field, count in negative_checks.items():
        if count > 0:
            print(f"⚠ {field}: {count} non-positive values")
        else:
            print(f"✓ {field}: All values are positive")
    
    # Check discount logic
    invalid_discounts = (df['discount_percent'] > 1).sum()
    if invalid_discounts > 0:
        print(f"⚠ discount_percent: {invalid_discounts} values > 100%")
    else:
        print("✓ discount_percent: All values are valid")
    
    return df

# Run quality check
df_clean = comprehensive_data_quality_check(df)

# Display sample data
print("\nSample data:")
print(df_clean.head())

## 3. Exploratory Data Analysis

In [ ]:
# Comprehensive EDA
def perform_eda(df):
    print("EXPLORATORY DATA ANALYSIS")
    print("=" * 40)
    
    # Basic statistics
    print("Basic Statistics:")
    print(df.describe())
    print()
    
    # Category distribution
    print("Category Distribution:")
    category_stats = df['category'].value_counts()
    print(category_stats)
    print()
    
    # Revenue by category
    revenue_by_category = df.groupby('category')['total_amount'].sum().sort_values(ascending=False)
    print("Revenue by Category:")
    print(revenue_by_category.round(2))
    print()
    
    # Customer demographics
    print("Customer Demographics:")
    print(f"Age distribution: {df['customer_age'].describe()}")
    print(f"Gender distribution: {df['customer_gender'].value_counts()}")
    print()
    
    # Payment methods
    print("Payment Method Distribution:")
    print(df['payment_method'].value_counts())
    print()
    
    # Time-based patterns
    print("Time-based Patterns:")
    hourly_sales = df.groupby('hour')['total_amount'].sum()
    print(f"Peak sales hour: {hourly_sales.idxmax()} ({hourly_sales.max():.2f})")
    
    daily_sales = df.groupby('day_of_week')['total_amount'].sum()
    days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    print(f"Peak sales day: {days[daily_sales.idxmax()]} ({daily_sales.max():.2f})")
    
    return df

# Perform EDA
df_analyzed = perform_eda(df_clean)

# Create visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Category distribution
df['category'].value_counts().plot(kind='bar', ax=axes[0, 0], title='Transactions by Category')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Revenue by category
revenue_by_cat = df.groupby('category')['total_amount'].sum()
revenue_by_cat.plot(kind='bar', ax=axes[0, 1], title='Revenue by Category')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Price distribution
df['unit_price'].hist(bins=50, ax=axes[0, 2], title='Unit Price Distribution')
axes[0, 2].set_xlabel('Unit Price')

# 4. Sales by hour
hourly_sales = df.groupby('hour')['total_amount'].sum()
hourly_sales.plot(kind='line', ax=axes[1, 0], title='Sales by Hour of Day', marker='o')
axes[1, 0].set_xlabel('Hour')

# 5. Customer age distribution
df['customer_age'].hist(bins=30, ax=axes[1, 1], title='Customer Age Distribution')
axes[1, 1].set_xlabel('Age')

# 6. Rating distribution
df['customer_rating'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 2], title='Customer Rating Distribution')
axes[1, 2].set_xlabel('Rating')

plt.tight_layout()
plt.show()

## 4. Advanced Analytics

In [ ]:
# Time series analysis
def time_series_analysis(df):
    print("TIME SERIES ANALYSIS")
    print("=" * 30)
    
    # Daily sales trend
    daily_sales = df.groupby(df['timestamp'].dt.date).agg({
        'total_amount': 'sum',
        'transaction_id': 'count'
    }).rename(columns={'transaction_id': 'transaction_count'})
    
    # Add moving averages
    daily_sales['revenue_ma_7'] = daily_sales['total_amount'].rolling(window=7).mean()
    daily_sales['revenue_ma_30'] = daily_sales['total_amount'].rolling(window=30).mean()
    
    # Monthly analysis
    monthly_sales = df.groupby(df['timestamp'].dt.to_period('M')).agg({
        'total_amount': 'sum',
        'transaction_id': 'count',
        'customer_id': 'nunique'
    }).rename(columns={
        'transaction_id': 'transactions',
        'customer_id': 'unique_customers'
    })
    
    monthly_sales['avg_order_value'] = monthly_sales['total_amount'] / monthly_sales['transactions']
    
    print("Monthly Performance:")
    print(monthly_sales.round(2))
    
    # Seasonal analysis
    seasonal_analysis = df.groupby('month').agg({
        'total_amount': ['sum', 'mean', 'count']
    })
    seasonal_analysis.columns = ['total_revenue', 'avg_revenue', 'transaction_count']
    
    print("\nSeasonal Patterns:")
    print(seasonal_analysis.round(2))
    
    return daily_sales, monthly_sales, seasonal_analysis

# Customer segmentation
def customer_segmentation(df):
    print("\nCUSTOMER SEGMENTATION")
    print("=" * 30)
    
    # RFM Analysis (Recency, Frequency, Monetary)
    current_date = df['timestamp'].max()
    
    rfm = df.groupby('customer_id').agg({
        'timestamp': lambda x: (current_date - x.max()).days,  # Recency
        'transaction_id': 'count',  # Frequency
        'total_amount': 'sum'  # Monetary
    }).rename(columns={
        'timestamp': 'recency',
        'transaction_id': 'frequency',
        'total_amount': 'monetary'
    })
    
    # Create RFM scores
    rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1])
    rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
    rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])
    
    # Combine scores
    rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)
    
    # Customer segments
    def segment_customers(row):
        if row['rfm_score'] in ['555', '554', '544', '545', '454', '455', '445']:
            return 'Champions'
        elif row['rfm_score'] in ['543', '444', '435', '355', '354', '345', '344', '335']:
            return 'Loyal Customers'
        elif row['rfm_score'] in ['553', '551', '552', '541', '542', '533', '532', '531', '452', '451']:
            return 'Potential Loyalists'
        elif row['rfm_score'] in ['512', '511', '422', '421', '412', '411', '311']:
            return 'New Customers'
        elif row['rfm_score'] in ['155', '154', '144', '214', '215', '115', '114']:
            return 'Cannot Lose Them'
        elif row['rfm_score'] in ['155', '254', '245']:
            return 'At Risk'
        else:
            return 'Others'
    
    rfm['segment'] = rfm.apply(segment_customers, axis=1)
    
    print("Customer Segments:")
    segment_summary = rfm['segment'].value_counts()
    print(segment_summary)
    
    # Segment characteristics
    segment_chars = rfm.groupby('segment').agg({
        'recency': 'mean',
        'frequency': 'mean',
        'monetary': 'mean'
    }).round(2)
    
    print("\nSegment Characteristics:")
    print(segment_chars)
    
    return rfm

# Product analysis
def product_analysis(df):
    print("\nPRODUCT ANALYSIS")
    print("=" * 20)
    
    # Product performance
    product_performance = df.groupby(['category', 'product_name']).agg({
        'total_amount': ['sum', 'mean', 'count'],
        'customer_rating': 'mean',
        'quantity': 'sum'
    })
    
    product_performance.columns = ['total_revenue', 'avg_revenue', 'transactions', 'avg_rating', 'total_quantity']
    product_performance = product_performance.sort_values('total_revenue', ascending=False)
    
    print("Top 10 Products by Revenue:")
    print(product_performance.head(10).round(2))
    
    # Category performance
    category_performance = df.groupby('category').agg({
        'total_amount': ['sum', 'mean'],
        'customer_rating': 'mean',
        'quantity': 'sum',
        'customer_id': 'nunique'
    })
    
    category_performance.columns = ['total_revenue', 'avg_order_value', 'avg_rating', 'total_quantity', 'unique_customers']
    
    print("\nCategory Performance:")
    print(category_performance.round(2))
    
    return product_performance, category_performance

# Run advanced analytics
daily_sales, monthly_sales, seasonal_analysis = time_series_analysis(df)
rfm_analysis = customer_segmentation(df)
product_perf, category_perf = product_analysis(df)

## 5. Business Intelligence Dashboard

In [ ]:
# Create comprehensive dashboard
def create_business_dashboard(df, daily_sales, rfm_analysis):
    fig = plt.figure(figsize=(20, 16))
    
    # 1. Daily revenue trend
    ax1 = plt.subplot(3, 4, (1, 2))
    daily_sales['total_amount'].plot(ax=ax1, title='Daily Revenue Trend', color='blue', alpha=0.7)
    daily_sales['revenue_ma_7'].plot(ax=ax1, color='red', label='7-day MA')
    daily_sales['revenue_ma_30'].plot(ax=ax1, color='green', label='30-day MA')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Category revenue pie chart
    ax2 = plt.subplot(3, 4, 3)
    category_revenue = df.groupby('category')['total_amount'].sum()
    ax2.pie(category_revenue.values, labels=category_revenue.index, autopct='%1.1f%%')
    ax2.set_title('Revenue by Category')
    
    # 3. Customer segments
    ax3 = plt.subplot(3, 4, 4)
    segment_counts = rfm_analysis['segment'].value_counts()
    ax3.bar(range(len(segment_counts)), segment_counts.values)
    ax3.set_xticks(range(len(segment_counts)))
    ax3.set_xticklabels(segment_counts.index, rotation=45, ha='right')
    ax3.set_title('Customer Segments')
    
    # 4. Hourly sales pattern
    ax4 = plt.subplot(3, 4, 5)
    hourly_pattern = df.groupby('hour')['total_amount'].sum()
    ax4.plot(hourly_pattern.index, hourly_pattern.values, marker='o')
    ax4.set_title('Sales by Hour')
    ax4.set_xlabel('Hour of Day')
    ax4.grid(True, alpha=0.3)
    
    # 5. Monthly sales comparison
    ax5 = plt.subplot(3, 4, 6)
    monthly_revenue = df.groupby('month')['total_amount'].sum()
    ax5.bar(monthly_revenue.index, monthly_revenue.values)
    ax5.set_title('Monthly Revenue')
    ax5.set_xlabel('Month')
    
    # 6. Price vs Rating scatter
    ax6 = plt.subplot(3, 4, 7)
    sample_data = df.sample(1000)  # Sample for performance
    ax6.scatter(sample_data['unit_price'], sample_data['customer_rating'], alpha=0.6)
    ax6.set_title('Price vs Rating')
    ax6.set_xlabel('Unit Price')
    ax6.set_ylabel('Rating')
    
    # 7. Payment method distribution
    ax7 = plt.subplot(3, 4, 8)
    payment_dist = df['payment_method'].value_counts()
    ax7.bar(payment_dist.index, payment_dist.values)
    ax7.set_title('Payment Methods')
    ax7.tick_params(axis='x', rotation=45)
    
    # 8. Age group analysis
    ax8 = plt.subplot(3, 4, 9)
    df['age_group'] = pd.cut(df['customer_age'], bins=[0, 25, 35, 45, 55, 100], 
                            labels=['18-25', '26-35', '36-45', '46-55', '55+'])
    age_revenue = df.groupby('age_group')['total_amount'].sum()
    ax8.bar(age_revenue.index, age_revenue.values)
    ax8.set_title('Revenue by Age Group')
    ax8.tick_params(axis='x', rotation=45)
    
    # 9. Weekend vs Weekday
    ax9 = plt.subplot(3, 4, 10)
    weekend_comparison = df.groupby('is_weekend')['total_amount'].sum()
    ax9.bar(['Weekday', 'Weekend'], weekend_comparison.values)
    ax9.set_title('Weekend vs Weekday Sales')
    
    # 10. Top products
    ax10 = plt.subplot(3, 4, 11)
    top_products = df.groupby('product_name')['total_amount'].sum().nlargest(10)
    ax10.barh(range(len(top_products)), top_products.values)
    ax10.set_yticks(range(len(top_products)))
    ax10.set_yticklabels(top_products.index)
    ax10.set_title('Top 10 Products')
    
    # 11. Discount impact
    ax11 = plt.subplot(3, 4, 12)
    df['discount_range'] = pd.cut(df['discount_percent'], bins=5)
    discount_impact = df.groupby('discount_range')['customer_rating'].mean()
    ax11.plot(range(len(discount_impact)), discount_impact.values, marker='o')
    ax11.set_title('Discount vs Rating')
    ax11.set_xticks(range(len(discount_impact)))
    ax11.set_xticklabels([f'{i:.1f}-{i+0.1:.1f}' for i in np.arange(0, 0.5, 0.1)], rotation=45)
    
    plt.suptitle('E-commerce Business Intelligence Dashboard', fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()

# Create dashboard
create_business_dashboard(df, daily_sales, rfm_analysis)

## 6. Automated Reporting System

In [ ]:
# Automated report generation
class BusinessReportGenerator:
    def __init__(self, df):
        self.df = df
        self.report_date = datetime.now()
    
    def generate_executive_summary(self):
        """Generate executive summary"""
        total_revenue = self.df['total_amount'].sum()
        total_transactions = len(self.df)
        unique_customers = self.df['customer_id'].nunique()
        avg_order_value = self.df['total_amount'].mean()
        
        summary = f"""
        EXECUTIVE SUMMARY
        ================
        Report Date: {self.report_date.strftime('%Y-%m-%d %H:%M:%S')}
        
        KEY METRICS:
        • Total Revenue: ${total_revenue:,.2f}
        • Total Transactions: {total_transactions:,}
        • Unique Customers: {unique_customers:,}
        • Average Order Value: ${avg_order_value:.2f}
        • Customer Retention Rate: {(total_transactions/unique_customers):.2f} orders per customer
        """
        return summary
    
    def generate_category_insights(self):
        """Generate category performance insights"""
        category_stats = self.df.groupby('category').agg({
            'total_amount': ['sum', 'mean', 'count'],
            'customer_rating': 'mean'
        })
        
        category_stats.columns = ['revenue', 'avg_order', 'transactions', 'avg_rating']
        category_stats = category_stats.sort_values('revenue', ascending=False)
        
        top_category = category_stats.index[0]
        top_revenue = category_stats.iloc[0]['revenue']
        
        insights = f"""
        CATEGORY PERFORMANCE
        ===================
        Top Performing Category: {top_category} (${top_revenue:,.2f})
        
        Category Breakdown:
        {category_stats.round(2).to_string()}
        """
        return insights
    
    def generate_customer_insights(self, rfm_data):
        """Generate customer insights"""
        segment_distribution = rfm_data['segment'].value_counts()
        champions_pct = (segment_distribution.get('Champions', 0) / len(rfm_data)) * 100
        
        insights = f"""
        CUSTOMER INSIGHTS
        ================
        Total Customers Analyzed: {len(rfm_data):,}
        Champions (Top Customers): {segment_distribution.get('Champions', 0):,} ({champions_pct:.1f}%)
        
        Customer Segments:
        {segment_distribution.to_string()}
        
        RECOMMENDATIONS:
        • Focus retention efforts on Champions and Loyal Customers
        • Develop win-back campaigns for At Risk customers
        • Create onboarding programs for New Customers
        """
        return insights
    
    def generate_time_insights(self):
        """Generate time-based insights"""
        # Peak hours
        hourly_sales = self.df.groupby('hour')['total_amount'].sum()
        peak_hour = hourly_sales.idxmax()
        
        # Peak days
        daily_sales = self.df.groupby('day_of_week')['total_amount'].sum()
        days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        peak_day = days[daily_sales.idxmax()]
        
        # Seasonal patterns
        monthly_sales = self.df.groupby('month')['total_amount'].sum()
        peak_month = monthly_sales.idxmax()
        
        insights = f"""
        TIME-BASED INSIGHTS
        ==================
        Peak Sales Hour: {peak_hour}:00
        Peak Sales Day: {peak_day}
        Peak Sales Month: {peak_month}
        
        Weekend vs Weekday Performance:
        {self.df.groupby('is_weekend')['total_amount'].sum().to_string()}
        
        RECOMMENDATIONS:
        • Schedule marketing campaigns during peak hours ({peak_hour}:00)
        • Increase inventory on {peak_day}s
        • Prepare for seasonal demand in month {peak_month}
        """
        return insights
    
    def generate_anomaly_alerts(self):
        """Generate anomaly alerts"""
        alerts = []
        
        # Check for unusual patterns
        avg_rating = self.df['customer_rating'].mean()
        if avg_rating < 3.5:
            alerts.append(f"⚠ Low average customer rating: {avg_rating:.2f}")
        
        # Check for high discount usage
        avg_discount = self.df['discount_percent'].mean()
        if avg_discount > 0.3:
            alerts.append(f"⚠ High average discount rate: {avg_discount:.1%}")
        
        # Check for price outliers
        price_q99 = self.df['unit_price'].quantile(0.99)
        high_price_count = (self.df['unit_price'] > price_q99).sum()
        if high_price_count > len(self.df) * 0.02:
            alerts.append(f"⚠ Unusual number of high-priced items: {high_price_count}")
        
        if alerts:
            alert_text = "\n        ".join(alerts)
            return f"""
        ANOMALY ALERTS
        ==============
        {alert_text}
        """
        else:
            return """
        ANOMALY ALERTS
        ==============
        ✓ No anomalies detected
        """
    
    def generate_full_report(self, rfm_data):
        """Generate complete business report"""
        report = """
        ╔══════════════════════════════════════════════════════════════╗
        ║                    BUSINESS INTELLIGENCE REPORT              ║
        ╚══════════════════════════════════════════════════════════════╝
        """
        
        report += self.generate_executive_summary()
        report += self.generate_category_insights()
        report += self.generate_customer_insights(rfm_data)
        report += self.generate_time_insights()
        report += self.generate_anomaly_alerts()
        
        return report
    
    def save_report(self, rfm_data, filename=None):
        """Save report to file"""
        if filename is None:
            filename = f"business_report_{self.report_date.strftime('%Y%m%d_%H%M%S')}.txt"
        
        report = self.generate_full_report(rfm_data)
        
        with open(filename, 'w') as f:
            f.write(report)
        
        print(f"Report saved to {filename}")
        return report

# Generate automated report
report_generator = BusinessReportGenerator(df)
full_report = report_generator.generate_full_report(rfm_analysis)
print(full_report)

# Save report
report_generator.save_report(rfm_analysis)

## 7. Export and Documentation

In [ ]:
# Export all results
def export_analysis_results(df, rfm_analysis, daily_sales, monthly_sales):
    """Export all analysis results to various formats"""
    
    # Create Excel workbook with multiple sheets
    with pd.ExcelWriter('ecommerce_analysis_complete.xlsx', engine='openpyxl') as writer:
        # Raw data sample
        df.head(1000).to_excel(writer, sheet_name='Raw_Data_Sample', index=False)
        
        # Summary statistics
        df.describe().to_excel(writer, sheet_name='Summary_Statistics')
        
        # Category analysis
        category_analysis = df.groupby('category').agg({
            'total_amount': ['sum', 'mean', 'count'],
            'customer_rating': 'mean',
            'quantity': 'sum'
        })
        category_analysis.to_excel(writer, sheet_name='Category_Analysis')
        
        # Product analysis
        product_analysis = df.groupby(['category', 'product_name']).agg({
            'total_amount': ['sum', 'mean'],
            'customer_rating': 'mean',
            'quantity': 'sum'
        }).head(50)
        product_analysis.to_excel(writer, sheet_name='Top_Products')
        
        # Customer segmentation
        rfm_analysis.to_excel(writer, sheet_name='Customer_Segmentation')
        
        # Time series data
        daily_sales.to_excel(writer, sheet_name='Daily_Sales')
        monthly_sales.to_excel(writer, sheet_name='Monthly_Sales')
        
        # Customer demographics
        demographics = df.groupby(['customer_gender', 'age_group']).agg({
            'total_amount': ['sum', 'mean'],
            'customer_id': 'nunique'
        })
        demographics.to_excel(writer, sheet_name='Demographics')
    
    print("✓ Complete analysis exported to ecommerce_analysis_complete.xlsx")
    
    # Export key datasets to CSV
    df.to_csv('ecommerce_transactions.csv', index=False)
    rfm_analysis.to_csv('customer_segmentation.csv')
    daily_sales.to_csv('daily_sales_trend.csv')
    
    print("✓ Key datasets exported to CSV files")
    
    # Create data dictionary
    data_dictionary = {
        'Column': df.columns,
        'Data_Type': [str(dtype) for dtype in df.dtypes],
        'Description': [
            'Unique transaction identifier',
            'Unique customer identifier', 
            'Product name',
            'Product category',
            'Quantity purchased',
            'Price per unit',
            'Discount percentage applied',
            'Transaction timestamp',
            'Customer age',
            'Customer gender (M/F)',
            'Payment method used',
            'Shipping cost',
            'Customer rating (1-5)',
            'Discount amount in currency',
            'Total transaction amount',
            'Month of transaction',
            'Day of week (0=Monday)',
            'Hour of transaction',
            'Whether transaction was on weekend',
            'Customer age group'
        ]
    }
    
    data_dict_df = pd.DataFrame(data_dictionary)
    data_dict_df.to_csv('data_dictionary.csv', index=False)
    
    print("✓ Data dictionary created")
    
    return True

# Export all results
export_success = export_analysis_results(df, rfm_analysis, daily_sales, monthly_sales)

# Create project summary
project_summary = f"""
E-COMMERCE DATA ANALYSIS PROJECT SUMMARY
=======================================

PROJECT OVERVIEW:
This capstone project demonstrates a complete end-to-end data analysis pipeline
using pandas and related libraries for e-commerce business intelligence.

DATASET CHARACTERISTICS:
• Total Transactions: {len(df):,}
• Date Range: {df['timestamp'].min()} to {df['timestamp'].max()}
• Unique Customers: {df['customer_id'].nunique():,}
• Product Categories: {df['category'].nunique()}
• Total Revenue: ${df['total_amount'].sum():,.2f}

ANALYSIS COMPONENTS:
1. Data Quality Assessment
2. Exploratory Data Analysis (EDA)
3. Time Series Analysis
4. Customer Segmentation (RFM Analysis)
5. Product Performance Analysis
6. Business Intelligence Dashboard
7. Automated Reporting System

KEY INSIGHTS:
• Top Category: {df.groupby('category')['total_amount'].sum().idxmax()}
• Peak Sales Hour: {df.groupby('hour')['total_amount'].sum().idxmax()}:00
• Average Order Value: ${df['total_amount'].mean():.2f}
• Customer Satisfaction: {df['customer_rating'].mean():.2f}/5.0

TECHNICAL SKILLS DEMONSTRATED:
• Data cleaning and preprocessing
• Advanced pandas operations (groupby, pivot, merge, etc.)
• Time series analysis and resampling
• Statistical analysis and outlier detection
• Data visualization with matplotlib/seaborn
• Custom function development
• Automated reporting and export functionality
• Memory optimization and performance tuning

FILES GENERATED:
• ecommerce_analysis_complete.xlsx (Complete analysis workbook)
• ecommerce_transactions.csv (Raw transaction data)
• customer_segmentation.csv (RFM analysis results)
• daily_sales_trend.csv (Time series data)
• data_dictionary.csv (Data documentation)
• business_report_[timestamp].txt (Automated business report)

This project showcases production-ready data analysis capabilities suitable for
real-world business intelligence and decision-making scenarios.
"""

print(project_summary)

# Save project summary
with open('project_summary.txt', 'w') as f:
    f.write(project_summary)

print("\n" + "="*60)
print("CONGRATULATIONS! 🎉")
print("You have successfully completed the 10-day Pandas Tutorial Series!")
print("You now have comprehensive skills in data analysis with pandas.")
print("="*60)